In [25]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from matplotlib import pyplot as plt
from IPython import display

In [26]:
DATASET_PATH = './dataset'

images = []
for name in os.listdir(DATASET_PATH):
    img = load_img(f"{DATASET_PATH}/{name}", target_size=(32, 32, 3))
    img = img_to_array(img)
    images.append(img)

images = np.array(images)
images = (images - 127.5) / 127.5

print(len(images))

FileNotFoundError: [Errno 2] No such file or directory: './dataset'

In [ ]:
# Hyperparameter
EPOCHS = 100
NOISE_DIM = 100
BATCH_SIZE = 20
BUFFER_SIZE = 60000
LEARNING_RATE = 0.0001

In [ ]:
generator_model = tf.keras.Sequential([
    layers.Dense(8 * 8 * 256, use_bias=False, input_shape=(100,)),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Reshape((8, 8, 256)),
    layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), use_bias=False, padding='same'),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), use_bias=False, padding='same'),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), use_bias=False, padding='same', activation='tanh')
])

sample_noise = tf.random.normal([1, NOISE_DIM])
generated_sample = generator_model(sample_noise)
plt.imshow((generated_sample[0] + 1.0) / 2.0)
plt.show()

In [ ]:
discriminator_model = tf.keras.Sequential([
    layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=(32, 32, 3)),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(32, (5, 5), strides=(2, 2), padding='same'),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(1)
])

print(discriminator_model(generated_sample, training=False))

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
def get_generator_loss(fake_image):
    return cross_entropy(tf.ones_like(fake_image), fake_image)

def get_discriminator_loss(fake, real):
    fake_loss = cross_entropy(tf.zeros_like(fake), fake)
    real_loss = cross_entropy(tf.ones_like(real), real)
    return fake_loss + real_loss

In [ ]:
noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

gen_optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)
disc_optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

@tf.function
def train_step(batch_image):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator_model(noise, training=True)

        disc_fake = discriminator_model(generated_images, training=True)
        disc_real = discriminator_model(batch_image, training=True)
        gen_loss = get_generator_loss(disc_fake)
        disc_loss = get_discriminator_loss(disc_fake, disc_real)

    gen_gradient = gen_tape.gradient(gen_loss, generator_model.trainable_variables)
    disc_gradient = disc_tape.gradient(disc_loss, discriminator_model.trainable_variables)

    gen_optimizer.apply_gradients(zip(gen_gradient, generator_model.trainable_variables))
    disc_optimizer.apply_gradients(zip(disc_gradient, discriminator_model.trainable_variables))


In [ ]:
def generate_images(input_img):
    prediction = generator_model(input_img, training=False)

    plt.figure(figsize=(4, 4))
    for i in range(input_img.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow((prediction[i] + 1.0) / 2.0)
        plt.axis('off')
    plt.show()

In [ ]:
def train(dataset, epochs):
    seed = tf.random.normal([16, NOISE_DIM])
    for epoch in range(epochs):
        for batch in dataset:
            train_step(batch)
        display.clear_output(wait=True)
        print(f"EPOCH {epoch + 1}")
        generate_images(seed)

    display.clear_output(wait=True)
    generate_images(seed)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(images)
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
train(train_dataset, EPOCHS)